# 8.5 Final Model Selection and Deployment

## Course 3: Machine Learning for Higher Education — Advanced Applications

The institution has always utilized survey data to get the pulse of student interests, preferences and priorities. Now they realize that survey data can also be used to enhance unsupervised and supervised machine learning models. In this notebook, we'll recap the model selection journey and deploy the survey-enhanced xgboost model to new data.

## Learning Objectives

1.  **Understand** the criteria for selecting a final machine learning model for deployment, moving beyond mere accuracy to consider actionability, explainability, and institutional fit.
2.  **Learn** the end-to-end process of deploying a machine learning model, encompassing data preparation, prediction generation, and the application of practical outreach thresholds.
3.  **Develop** effective communication strategies for presenting model outputs to diverse stakeholders, including the creation of interpretable risk bands and advisor-facing outreach lists.
4.  **Identify** crucial considerations for responsible AI deployment, such as addressing ethical implications, ensuring privacy, establishing human oversight, and planning for ongoing monitoring.
5.  **Recognize** the essential components of a robust deployable model, including the model artifact, feature schema, model card, and a comprehensive deployment checklist.

## 1. Final Model Selection Logic

Recall that the advanced comparison in **8.4** compares four models:

| Model | Feature Set | Main Strength |
|---|---|---|
| Regularized Logistic Regression | Administrative/student-record features | Most interpretable and easiest to explain |
| Random Forest | Administrative/student-record features | Strong nonlinear baseline with good robustness |
| XGBoost | Administrative/student-record features | Strong predictive performance on tabular data |
| Survey-Enhanced XGBoost | Academic + demographic + survey/text features | Best multimodal model because it uses both records and student voice |

For this final deployment notebook, the **recommended best overall model** is:

## Survey-Enhanced XGBoost

This recommendation is based on the logic developed across Modules 8.1–8.4:

- 8.1 established why survey data can add student experience signals that administrative records miss.
- 8.2 showed how survey/text features can reveal student personas and hidden patterns.
- 8.3 trained a survey-enhanced XGBoost model for 3rd-semester status.
- 8.4 compared that model against the original 4.1 model families.

The deployment question is not simply, **Which model has the highest accuracy?**  
The stronger question is, **Which model provides the best balance of prediction, actionability, and institutional explanation?**

## 2. Setup

Expected files:

- `training.csv`
- `testing.csv`
- `ML_SURVEY_MASTER_TRAIN.csv`
- `ML_SURVEY_MASTER_TEST.csv`

The two survey master files are the preferred deployment inputs because they include the engineered survey and text features from Module 7 and Module 8.

In [1]:
# If needed in Colab, uncomment the next line:
# !pip -q install xgboost

import numpy as np
import pandas as pd
import warnings
import time
import joblib
import json
from pathlib import Path

warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, brier_score_loss, log_loss
)

import pickle

RANDOM_STATE = 2
np.random.seed(RANDOM_STATE)

## 3. Data Loading Helper

The original Course 3 notebooks sometimes use `../data/`, while the Module 8 notebooks often use a Google Drive path. The helper below checks several common locations.

Update `candidate_dirs` if your files live somewhere else.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
data_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'

# Load training deploy data
df_deploy = pd.read_csv(f'{data_filepath}Deploy_Survey_Data.csv')
pd.set_option('display.max_columns', None)
df_deploy

,SID,HS_GPA,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,GENDER_Female,GENDER_Male,RACE_ETHNICITY_Asian,RACE_ETHNICITY_Black or African American,RACE_ETHNICITY_Hispanic,RACE_ETHNICITY_Nonresident alien,RACE_ETHNICITY_Other,RACE_ETHNICITY_Two or More Races,RACE_ETHNICITY_White,FIRST_GEN_STATUS_Continuing Generation,FIRST_GEN_STATUS_First Generation,FIRST_GEN_STATUS_Unknown,TEXT_PC1,TEXT_PC2,TEXT_PC3,TEXT_PC4,TEXT_PC5,TEXT_PC6,TEXT_PC7,TEXT_PC8,TEXT_PC9,TEXT_PC10,TEXT_PC11,TEXT_PC12,TEXT_PC13,TEXT_PC14,TEXT_PC15,TEXT_PC16,TEXT_PC17,TEXT_PC18,TEXT_PC19,TEXT_PC20,TEXT_PC21,TEXT_PC22,TEXT_PC23,TEXT_PC24,TEXT_PC25,TEXT_PC26,TEXT_PC27,TEXT_PC28,TEXT_PC29,TEXT_PC30,TEXT_PC31,TEXT_PC32,TEXT_PC33
0,WBF4NV3BB,0.358974,0.819512,0.517241,0.200000,0.500000,0.993514,0.073885,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.048116,-0.028699,0.028671,-0.043197,-0.052049,-0.089073,0.013749,0.040051,-0.052713,-0.038672,0.088200,0.020428,-0.016728,0.042960,0.040951,0.025467,0.052907,-0.050895,0.142338,0.341808,-0.267093,-0.574248,-0.472475,-0.139799,-0.040449,0.044541,-0.013967,-0.038628,-0.148582,-0.220729,-0.067357,-0.186668,-0.114515
1,3S0KTAEMM,0.475641,0.146341,1.000000,0.785714,0.000000,0.600581,-0.575244,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.040111,-0.037142,0.023370,-0.046080,-0.053456,-0.071037,0.093725,-0.024801,-0.054805,-0.061195,0.082241,0.017650,-0.010052,0.069868,0.001529,-0.114869,0.140442,-0.044172,-0.010071,0.126500,-0.199042,0.215520,-0.062881,-0.014371,0.050376,-0.079815,-0.052404,0.048873,0.034063,0.053589,0.036874,0.409665,-0.300937
2,8T4XQG9AB,0.455769,0.956098,0.985222,0.200000,0.000000,0.993514,0.723013,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.022823,0.038930,-0.110642,-0.261648,-0.404762,0.166457,-0.422280,-0.230225,-0.246531,-0.210596,0.024881,-0.102128,0.235045,-0.043970,0.099240,0.036394,-0.228109,0.134172,-0.351530,0.097190,-0.000110,0.046869,-0.059637,-0.010378,0.034216,-0.016542,0.009989,-0.037470,-0.061510,0.033594,-0.001905,0.003635,0.009945
3,HRQ62780I,0.575641,0.893058,0.000000,0.000000,1.000000,0.207648,-1.548937,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.533243,0.672777,0.077724,0.106012,0.081221,-0.163549,-0.052320,-0.020216,-0.022763,0.031270,-0.097379,-0.138133,0.052116,0.019970,-0.004847,-0.013348,-0.031328,-0.167934,-0.002564,0.045380,0.148911,0.053017,-0.046300,0.012054,0.009986,-0.017804,0.006963,0.022131,0.006335,-0.047888,0.008383,-0.002609,-0.030070
4,LBX6AR928,0.664103,0.000000,0.000000,1.000000,1.000000,-3.721684,0.073885,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.026967,-0.043455,-0.015432,-0.024530,-0.046958,0.039956,0.030007,-0.026006,0.260144,0.086663,0.014947,0.218916,-0.149628,0.020357,-0.042139,0.042457,-0.022987,0.283396,0.009891,0.184869,0.253799,0.035041,-0.150616,-0.023328,0.387257,-0.039982,0.051341,-0.453401,0.490230,-0.085161,-0.019697,-0.015050,-0.034363
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,8HWVJ0EVY,0.478205,0.796748,0.775862,0.000000,0.000000,-1.364085,0.073885,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.013698,-0.068474,-0.021035,-0.023426,-0.035466,0.001712,0.030871,-0.041079,0.143729,0.083593,-0.014326,-0.001268,-0.141987,-0.010053,0.105943,-0.011520,-0.062175,0.020355,0.009481,0.038594,0.028546,0.011488,-0.052368,-0.055673,0.083696,-0.034614,0.032737,-0.086296,0.128788,-0.008828,-0.002994,0.028694,0.137763
119,CXJRDEXQ2,0.207692,0.939024,0.940439,0.000000,0.000000,-0.185286,-0.250680,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.007848,-0.130106,-0.189059,0.246978,-0.167925,0.084806,-0.229731,-0.196710,-0.444345,0.649987,-0.119538,0.228546,-0.009772,-0.008373,-0.058958,-0.024151,0.092911,-0.052471,0.093105,-0.061234,0.013429,-0.001524,0.000651,-0.008092,-0.022127,-0.025187,-0.003261,0.007316,0.007394,-0.013967,0.001865,-0.

As it will be critical to identify students by name, and reference raw data. Let's be sure to load the deploy data with those identifiers.

In [8]:
# Load training deploy data
df_deploy_names = pd.read_csv(f'{data_filepath}Deploy_Data_Other.csv')
pd.set_option('display.max_columns', None)
df_deploy_names

,SID,NAME,LAST_NAME,COHORT,RACE_ETHNICITY,GENDER,FIRST_GEN_STATUS,HS_GPA,HS_MATH_GPA,HS_ENGL_GPA,COLLEGE,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,UNITS_COMPLETED_1,UNITS_COMPLETED_2,DFW_UNITS_1,DFW_UNITS_2,DFW_UNITS_3,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,GRADE_POINTS_1,GRADE_POINTS_2
0,WBF4NV3BB,Reina,Smith,Fall 2023,White,Female,Continuing Generation,3.200,2.750,3.250,Business Administration,15,12,12,6,3,6,0,2.400000,1.500000,0.200000,0.500000,36.0,18.0
1,3S0KTAEMM,Ai,Xu,Fall 2023,Asian,Female,First Generation,3.382,3.250,4.125,General Studies,14,10,3,10,11,3,0,0.428571,2.900000,0.785714,0.000000,6.0,29.0
2,8T4XQG9AB,Naoko,Takahashi,Fall 2023,Asian,Female,Continuing Generation,3.351,2.750,2.300,General Studies,15,14,12,14,3,0,0,2.800000,2.857143,0.200000,0.000000,42.0,40.0
3,HRQ62780I,Esteban,Gomez,Fall 2023,Hispanic,Male,Continuing Generation,3.538,3.833,3.667,Engineering & Technology,13,7,13,0,0,13,8,2.615385,0.000000,0.000000,1.000000,34.0,0.0
4,LBX6AR928,Maria,Jones,Fall 2023,White,Female,Unknown,3.676,3.400,3.333,Education & Leadership,3,12,0,0,12,12,0,0.000000,0.000000,1.000000,1.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,8HWVJ0EVY,Angela,Kardishian,Fall 2023,White,Female,Unknown,3.386,3.250,3.500,Business Administration,9,12,9,12,6,0,0,2.333333,2.250000,0.000000,0.000000,21.0,27.0
119,CXJRDEXQ2,Valeria,Silva,Fall 2023,Hispanic,Female,Unknown,2.964,2.667,2.600,General Studies,12,11,12,11,3,0,0,2.750000,2.727273,0.000000,0.000000,33.0,30.0
120,6F9AWVIOG,Miguel,Martinez,Fall 2023,Hispanic,Male,Continuing Generation,3.808,3.600,3.600,General Studies,15,12,15,9,0,3,3,2.400000,2.083333,0.000000,0.250000,36.0,25.0
121,UQH18FU24,Noemi,Xu,Fall 2023,Asian,Female,First Generation,2.725,2.200,2.875,Business Administration,15,15,9,12,6,3,9,2.000000,2.200000,0.400000,0.200000,30.0,33.0


# 4. Pickle in the Best Model

In [9]:
# Load the model from the pickle file
model_filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Models/'
filename1 = f'{model_filepath}Survey_xgb_model.pkl'
survey_xgb_model = pickle.load(open(filename1, 'rb'))

filename2 = f'{model_filepath}kmeans_model.pkl'
kmeans_cluster_model = pickle.load(open(filename2, 'rb'))
# You can now use the loaded_model_pkl object for predictions or other tasks
print("Model loaded successfully from pickle file.")

Model loaded successfully from pickle file.


## 5. Deploy the Recommended Model to the Hold-Out Set

In a real institutional workflow, the hold-out set represents students the model did **not** see during training.

The deployment steps are:

1. Load the saved model and feature schema.
2. Generate a predicted probability for each student.
3. Convert probabilities into risk bands.
4. Share a carefully framed outreach list with advisors or student success teams.

In [12]:
import numpy as np
# Predicted probability of departure for each hold-out student
# Drop the 'SID' column as it is a string identifier and not a numerical feature for the model.
holdout_prob = survey_xgb_model.predict_proba(df_deploy.drop('SID', axis=1))[:, 1]

# Create an indicator variable that =1 for students predicted to depart and =0 if not
holdout_pred_default = (holdout_prob >= 0.50).astype(int)

holdout_scores = df_deploy.copy()
holdout_scores['departure_risk_score'] = holdout_prob
holdout_scores['predicted_departed_default_050'] = holdout_pred_default

holdout_scores

,SID,HS_GPA,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,GENDER_Female,GENDER_Male,RACE_ETHNICITY_Asian,RACE_ETHNICITY_Black or African American,RACE_ETHNICITY_Hispanic,RACE_ETHNICITY_Nonresident alien,RACE_ETHNICITY_Other,RACE_ETHNICITY_Two or More Races,RACE_ETHNICITY_White,FIRST_GEN_STATUS_Continuing Generation,FIRST_GEN_STATUS_First Generation,FIRST_GEN_STATUS_Unknown,TEXT_PC1,TEXT_PC2,TEXT_PC3,TEXT_PC4,TEXT_PC5,TEXT_PC6,TEXT_PC7,TEXT_PC8,TEXT_PC9,TEXT_PC10,TEXT_PC11,TEXT_PC12,TEXT_PC13,TEXT_PC14,TEXT_PC15,TEXT_PC16,TEXT_PC17,TEXT_PC18,TEXT_PC19,TEXT_PC20,TEXT_PC21,TEXT_PC22,TEXT_PC23,TEXT_PC24,TEXT_PC25,TEXT_PC26,TEXT_PC27,TEXT_PC28,TEXT_PC29,TEXT_PC30,TEXT_PC31,TEXT_PC32,TEXT_PC33,departure_risk_score,predicted_departed_default_050
0,WBF4NV3BB,0.358974,0.819512,0.517241,0.200000,0.500000,0.993514,0.073885,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.048116,-0.028699,0.028671,-0.043197,-0.052049,-0.089073,0.013749,0.040051,-0.052713,-0.038672,0.088200,0.020428,-0.016728,0.042960,0.040951,0.025467,0.052907,-0.050895,0.142338,0.341808,-0.267093,-0.574248,-0.472475,-0.139799,-0.040449,0.044541,-0.013967,-0.038628,-0.148582,-0.220729,-0.067357,-0.186668,-0.114515,0.369445,0
1,3S0KTAEMM,0.475641,0.146341,1.000000,0.785714,0.000000,0.600581,-0.575244,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.040111,-0.037142,0.023370,-0.046080,-0.053456,-0.071037,0.093725,-0.024801,-0.054805,-0.061195,0.082241,0.017650,-0.010052,0.069868,0.001529,-0.114869,0.140442,-0.044172,-0.010071,0.126500,-0.199042,0.215520,-0.062881,-0.014371,0.050376,-0.079815,-0.052404,0.048873,0.034063,0.053589,0.036874,0.409665,-0.300937,0.441571,0
2,8T4XQG9AB,0.455769,0.956098,0.985222,0.200000,0.000000,0.993514,0.723013,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.022823,0.038930,-0.110642,-0.261648,-0.404762,0.166457,-0.422280,-0.230225,-0.246531,-0.210596,0.024881,-0.102128,0.235045,-0.043970,0.099240,0.036394,-0.228109,0.134172,-0.351530,0.097190,-0.000110,0.046869,-0.059637,-0.010378,0.034216,-0.016542,0.009989,-0.037470,-0.061510,0.033594,-0.001905,0.003635,0.009945,0.215155,0
3,HRQ62780I,0.575641,0.893058,0.000000,0.000000,1.000000,0.207648,-1.548937,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.533243,0.672777,0.077724,0.106012,0.081221,-0.163549,-0.052320,-0.020216,-0.022763,0.031270,-0.097379,-0.138133,0.052116,0.019970,-0.004847,-0.013348,-0.031328,-0.167934,-0.002564,0.045380,0.148911,0.053017,-0.046300,0.012054,0.009986,-0.017804,0.006963,0.022131,0.006335,-0.047888,0.008383,-0.002609,-0.030070,0.634021,1
4,LBX6AR928,0.664103,0.000000,0.000000,1.000000,1.000000,-3.721684,0.073885,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.026967,-0.043455,-0.015432,-0.024530,-0.046958,0.039956,0.030007,-0.026006,0.260144,0.086663,0.014947,0.218916,-0.149628,0.020357,-0.042139,0.042457,-0.022987,0.283396,0.009891,0.184869,0.253799,0.035041,-0.150616,-0.023328,0.387257,-0.039982,0.051341,-0.453401,0.490230,-0.085161,-0.019697,-0.015050,-0.034363,0.883656,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,8HWVJ0EVY,0.478205,0.796748,0.775862,0.000000,0.000000,-1.364085,0.073885,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-0.013698,-0.068474,-0.021035,-0.023426,-0.035466,0.001712,0.030871,-0.041079,0.143729,0.083593,-0.014326,-0.001268,-0.141987,-0.010053,0.105943,-0.011520,-0.062175,0.020355,0.009481,0.038594,0.028546,0.011488,-0.052368,-0.055673,0.083696,-0.034614,0.032737,-0.086296,0.128788,-0.008828,-0.002994,0.028694,0.137763,0.321825,0
119,CXJRDEXQ2,0.207692,0.939024,0.940439,0.000000,0.000000,-0.185286,-0.250680,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.007848,-0.130106,-0.189059,0.246978,-0.167925,0.084806,-0.229731,-0.196710,-0.444345,0.649987,-0.119538,0.228546,-0.009772,-0.008373,-0.058958,-0.024151,0.092911,-0.052471,0

Let's take a look at some sumary statistics around departure for our deployment set:

In [13]:
# Add a stable row identifier if the dataset does not already include one.
if 'SID' not in holdout_scores.columns and 'STUDENT_ID' not in holdout_scores.columns:
    holdout_scores.insert(0, 'holdout_row_id', np.arange(len(holdout_scores)))

holdout_scores[['departure_risk_score', 'predicted_departed_default_050']].describe().round(4)

,departure_risk_score,predicted_departed_default_050
count,123.0000,123.0000
mean,0.4299,0.2927
std,0.2248,0.4569
min,0.1621,0.0000
25%,0.2516,0.0000
50%,0.3368,0.0000
75%,0.5895,1.0000
max,0.9095,1.0000


## 6. Choose a Practical Outreach Threshold

A probability threshold of `0.50` is mathematically simple, but it may not match advising capacity.

For deployment, institutions often choose a threshold based on one of these practical rules:

| Threshold Strategy | Meaning |
|---|---|
| Default threshold | Flag students with predicted probability ≥ 0.50 |
| Capacity threshold | Flag the top X% of students because that is how many advisors can contact |
| Recall-oriented threshold | Lower the threshold to catch more true departures, accepting more false positives |
| Precision-oriented threshold | Raise the threshold so outreach lists are smaller and more concentrated |

Below, we create a **capacity-based threshold** that flags the top 20% of hold-out students by predicted risk. We use the statistical concept of *quantiles* (generalized version of percentiles), which allow us to start with the desired percent and work backwards to find the relevant corresponding threshold.

In [14]:
capacity_share = 0.20  # Change this if advising capacity is larger or smaller.
capacity_threshold = float(np.quantile(holdout_prob, 1 - capacity_share))
holdout_scores['flag_top_20pct_capacity'] = (holdout_scores['departure_risk_score'] >= capacity_threshold).astype(int)

print(f'Capacity-based threshold for top {capacity_share:.0%}: {capacity_threshold:.4f}')
print('Number flagged:', int(holdout_scores['flag_top_20pct_capacity'].sum()))
print('Share flagged:', holdout_scores['flag_top_20pct_capacity'].mean().round(4))


Capacity-based threshold for top 20%: 0.6599
Number flagged: 25
Share flagged: 0.2033


## 7. Convert Risk Scores into Stakeholder-Friendly Risk Bands

Risk bands are easier to communicate than raw probabilities.

The exact labels should be reviewed by campus leadership and advising teams before use. Avoid stigmatizing language. In this notebook, we use:

- **Priority outreach**
- **Monitor/support**
- **Routine support**

These labels focus on the institutional action rather than labeling the student.

In [15]:
def assign_risk_band(score):
    if score >= np.quantile(holdout_prob, 0.80):
        return 'Priority outreach'
    elif score >= np.quantile(holdout_prob, 0.50):
        return 'Monitor/support'
    else:
        return 'Routine support'

holdout_scores['support_band'] = holdout_scores['departure_risk_score'].apply(assign_risk_band)

band_summary = (
    holdout_scores
    .groupby('support_band')
    .agg(
        students=('departure_risk_score', 'size'),
        avg_risk_score=('departure_risk_score', 'mean'),
    )
    .sort_values('avg_risk_score', ascending=False)
)

band_summary.round(4)

,students,avg_risk_score
support_band,,
Priority outreach,25,0.8136
Monitor/support,37,0.4536
Routine support,61,0.2582


In [16]:
fig = px.bar(
    band_summary.reset_index(),
    x='support_band',
    y='students',
    text='students',
    title='Hold-Out Students by Support Band',
    labels={'support_band': 'Support Band', 'students': 'Number of Students'}
)
fig.update_traces(textposition='outside')
fig.update_layout(height=450)
fig.show()

## 8. Create an Advisor-Facing Outreach List

The outreach list should contain only what advisors need to act. In production, avoid sharing unnecessary sensitive features.

Recommended columns:

- A student identifier
- Risk score
- Support band
- A few high-level academic indicators
- A note that the score is advisory and should not be used punitively

In [17]:
df_deploy_risk = pd.concat([df_deploy_names[['SID','NAME','LAST_NAME']],holdout_scores],axis=1)

In [18]:

recommended_context_cols = [
    'SID',
    'NAME',
    'LAST_NAME',
    'departure_risk_score',
    'support_band',
    'flag_top_20pct_capacity',
    'HS_GPA',
    'GPA_1',
    'GPA_2',
    'DFW_RATE_1',
    'DFW_RATE_2',
    'UNITS_ATTEMPTED_1',
    'UNITS_ATTEMPTED_2'
]



advisor_outreach_list = (
    df_deploy_risk[recommended_context_cols]
    .sort_values('departure_risk_score', ascending=False)
    .reset_index(drop=True)
)

advisor_outreach_list.head(123).round(4)

,SID,SID,NAME,LAST_NAME,departure_risk_score,support_band,flag_top_20pct_capacity,HS_GPA,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2
0,H1QR9PCB7,H1QR9PCB7,Isabella,Pena,0.9095,Priority outreach,1,0.3551,0.0263,0.0000,1.0000,1.0000,0.2076,-1.8735
1,57UJOLSG1,57UJOLSG1,Alicia,Delgado,0.9070,Priority outreach,1,0.5872,0.0000,0.0000,1.0000,1.0000,-1.3641,0.0739
2,WCNC1LLED,WCNC1LLED,Jose,Torres,0.9068,Priority outreach,1,0.2308,0.0000,0.0000,1.0000,1.0000,0.2076,-0.8998
3,IKOGCQ5TA,IKOGCQ5TA,Ximena,Curado,0.9022,Priority outreach,1,0.4378,0.0000,0.0000,1.0000,1.0000,0.2076,0.3984
4,AUBPNLPDT,AUBPNLPDT,Eduardo,Mendoza,0.8968,Priority outreach,1,0.3378,0.0683,0.0000,0.9000,1.0000,-0.9712,-0.8998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,O6D7UOZT7,O6D7UOZT7,Lola,Vargas,0.2059,Routine support,0,0.4641,0.7683,0.7692,0.2500,0.2308,-0.1853,0.3984
119,MTLR1ADIT,MTLR1ADIT,Amita,Singh,0.2030,Routine support,0,0.4949,0.8537,0.8276,0.0000,0.0000,-0.1853,1.0476
120,7E05NF8GC,7E05NF8GC,Andres,Tumko,0.1914,Routine support,0,0.7474,0.8878,0.9655,0.4000,0.0000,0.9935,1.0476
121,XFTZ73CLL,XFTZ73CLL,Naoko,Yamamoto,0.1880,Routine support,0,0.0987,0.6041,0.9814,0.2308,0.0000,0.2076,0.3984


## 9. Save the Model, Feature Schema, and Deployment Outputs

A deployable model is more than the model file. It should include:

- The model artifact
- The feature list/schema
- The risk scoring output
- A model card
- A note explaining the intended use and limitations

## 10. Model Card for Stakeholders

A deployable model is more than the model file. It should include:

- The model artifact
- The feature list/schema
- The risk scoring output
- A model card
- A note explaining the intended use and limitations

The following model card documents what the model is, what it is for, and what its limits are.

| Field | Description |
|:------|:-----------|
| **Model Name** | Survey-Enhanced Student Departure Risk Model v1.0 |
| **Model Type** | XGBoost binary classifier |
| **Task** | Binary classification: predict 3rd semester departure |
| **Training Data** | CSULB first-time freshmen cohorts |
| **Features** | HS GPA, college GPA, DFW rates, demographics, qualitative survey responses |
| **Performance (AUC)** | 0.8791 |
| **Intended Use** | Early warning system for academic advisors |
| **Limitations** | Trained on CSULB data; may not generalize to other institutions |
| **Ethical Considerations** | Low bias across demographic groups |
| **Retraining Schedule** | Annually, with each new cohort |
| **Owner** | Institutional Research and Analytics Department |

## 11. Plain-Language Explanation for Non-Technical Stakeholders

### What the model does

The model estimates which students in the hold-out set appear more likely to depart before the third semester. It uses information already prepared in the survey master matrix, including academic indicators and survey/text-derived signals.

### What the model does not do

The model does not determine a student's future. It does not explain causation. It should not be used to punish students, restrict opportunities, or make automatic decisions.

### How the institution should use it

The best use is supportive prioritization. Students in the highest support band should be considered for earlier advising, resource connection, or check-in communication.

### How to explain the risk score

A risk score means:

> “Based on patterns from prior students, this student looks similar to students who were more likely to depart. The score is a prompt for supportive outreach, not a judgment about the student.”

## 12. Deployment Checklist

Before using this model operationally, complete the following checklist. Note that some of these checks were performed as part of our model selection in section 8.4:

| Area | Check |
|---|---|
| Technical validation | Confirm model runs on new data without column mismatch errors |
| Hold-out performance | Review F1, recall, ROC-AUC, and precision-recall performance |
| Threshold choice | Align threshold with advising capacity and intervention design |
| Equity review | Compare performance and flag rates across student subgroups |
| Privacy review | Remove unnecessary sensitive fields from outreach files |
| Documentation | Save model card, feature schema, model artifact, and scoring output |
| Human oversight | Make clear that advisors make decisions, not the model |
| Monitoring | Track whether outreach improves outcomes and whether model drift occurs |
| Governance | Review with institutional data governance or analytics leadership |

## 13. Final Recommendation

In this notebook, we deployed the recommended final model - **Survey-Enhanced XGBoost**. This extends the original 4.1 comparison by adding survey and text-derived features to the strongest tree-based modeling family used in the course. This allows the model to use both administrative records and student voice.

The model has clear advantages from the institutional perspective; it produces an actionable risk score that can be translated into support bands for advising teams. The output is not meant to label students; it is meant to help the institution organize limited support capacity. Deploy the model as a **decision-support tool**, not a decision-making system.

## 14. Conclusion

You have learned more than a set of algorithms. You've experienced a powerful framework for creating an end to end decision support system that incorporates student background, experience and voice to strengthen predictive modeling. With this training, you are ready to apply this process to real higher ed challenges in an ethical way.  